In [0]:
CREATE OR REFRESH MATERIALIZED VIEW mavenmarket.gold.dim_customer
COMMENT 'Customer Dimension Table'
AS
SELECT
    customer_sk,
    customer_id,
    customer_acct_num AS account_number,
    TRIM(first_name) AS first_name,
    TRIM(last_name) AS last_name,
    CONCAT(TRIM(first_name), ' ', TRIM(last_name)) AS full_name,
    acct_open_date AS account_open_date,
    birthdate AS birth_date,

    -- Calculated Age based on birthdate
    FLOOR(DATEDIFF(CURRENT_DATE(), birthdate) / 365.25) AS age_years,
    TRIM(customer_address) AS street_address,
    TRIM(customer_city) AS city_name,
    TRIM(customer_state_province) AS state_province_code,
    TRIM(customer_country) AS country_name,
    TRIM(customer_postal_code) AS postal_code,
    TRIM(gender) AS gender_code,
    TRIM(marital_status) AS marital_status_code,
    TRIM(education) AS education_level,
    TRIM(occupation) AS occupation_name,
    TRIM(homeowner) AS homeowner_flag,

    -- Dynamic parsing for bounded ranges and open-ended '+' ranges
    CAST(
        CASE
            WHEN yearly_income LIKE '%+%' THEN
                TRY_CAST(REGEXP_EXTRACT(yearly_income, '\\$(\\d+)K', 1) AS DOUBLE) * 1000
            ELSE
                (
                    TRY_CAST(REGEXP_EXTRACT(yearly_income, '\\$(\\d+)K', 1) AS DOUBLE) +
                    TRY_CAST(REGEXP_EXTRACT(yearly_income, '-\\s*\\$(\\d+)K', 1) AS DOUBLE)
                ) / 2 * 1000
        END
        AS DECIMAL(18,2)
    ) AS yearly_income_amount,
    TRIM(yearly_income) AS yearly_income_range,

    -- CDC / SCD Type 2 tracking columns directly from Silver
    __START_AT AS start_at_timestamp,
    __END_AT AS end_at_timestamp,
    __END_AT IS NULL AS is_current_flag

FROM mavenmarket.silver.silver_customers
WHERE customer_id IS NOT NULL; 